# 02 | Initial Two-Stage Decision Prototype

**Historical exploratory implementation — not a validated subsidy policy.** This earlier notebook uses a different 155-feature master-table schema from the later 78-feature pipeline. The original feature list was recovered verbatim from the saved model-inspection output. The pickle is not needed in this edition because it was used only to obtain feature names.

The treatment labels are proxies inferred from payment records, not verified campaign assignments. The original outcome-model feature list includes payment-derived variables that reveal treatment; this is a central limitation, deliberately preserved rather than silently fixed.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

# Run from the repository root or notebooks/ directory.
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA = ROOT / "data"
PARQ = DATA / "parquet"
OUTPUT = ROOT / "outputs"
OUTPUT.mkdir(exist_ok=True)

import json
import lightgbm as lgb
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt
import seaborn as sns

df_train = pd.read_parquet(DATA / 'master_train_v2.parquet')
df_test = pd.read_parquet(DATA / 'master_test_v2.parquet')

## Define the first-stage groups

In [ ]:
import pandas as pd
import numpy as np

def add_coupon_features_v3(df):
    df_new = df.copy()


    df_new['coupon'] = df_new['last_plan_list_price'] - df_new['last_actual_paid']






    conditions = [
        (df_new['last_actual_paid'] == 0),
        (df_new['last_actual_paid'] > 0) & (df_new['coupon'] <= 0),
        (df_new['last_actual_paid'] > 0) & (df_new['coupon'] > 0)
    ]
    choices = ['Excluded', 'NoCoupon', 'Coupon']

    df_new['coupon_group'] = np.select(conditions, choices, default='Unknown')

    return df_new


df_train_updated = add_coupon_features_v3(df_train)


display(df_train_updated['coupon_group'].value_counts())

## Stage 1 | Propensity score matching

The original implementation matches on the propensity probability with a 0.3-standard-deviation caliper, allowing replacement. The ten matching covariates are narrower than the 155 outcome-model features. Good observed balance would not establish exchangeability.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler


pd.set_option('display.float_format', '{:.4f}'.format)

top_10_features = [
    'last_is_auto_renew', 'auto_renew_streak', 'last_payment_plan_days',
    'last_is_cancel', 'days_since_last_log', 'auto_renew_rate',
    'last_days_since_trans',
    '90d_num_25_count', 'total_cancellations', 'registered_via'
]


df_psm = df_train_updated[df_train_updated['coupon_group'].isin(['NoCoupon', 'Coupon'])].copy()


for col in top_10_features:
    df_psm[col] = pd.to_numeric(df_psm[col], errors='coerce').fillna(0)

X = df_psm[top_10_features]


scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)




ps_model = LogisticRegression(solver='lbfgs', max_iter=500, random_state=42)
ps_model.fit(X_scaled, df_psm['coupon_group'])


coupon_idx = list(ps_model.classes_).index('Coupon')
df_psm['ps_score'] = ps_model.predict_proba(X_scaled)[:, coupon_idx]





df_nocoupon = df_psm[df_psm['coupon_group'] == 'NoCoupon'].reset_index(drop=True)
df_coupon   = df_psm[df_psm['coupon_group'] == 'Coupon'].reset_index(drop=True)


caliper = 0.3 * df_psm['ps_score'].std()


nn = NearestNeighbors(n_neighbors=1, algorithm='ball_tree')
nn.fit(df_nocoupon[['ps_score']])


distances, indices = nn.kneighbors(df_coupon[['ps_score']])


valid_mask = distances.flatten() <= caliper


matched_coupon = df_coupon[valid_mask].copy()


matched_control_idx = indices.flatten()[valid_mask]
matched_control = df_nocoupon.iloc[matched_control_idx].copy()


matched_control['coupon_group'] = 'NoCoupon_Matched'



df_train_balanced = pd.concat([matched_coupon, matched_control]).reset_index(drop=True)
df_train_balanced['coupon_group'] = df_train_balanced['coupon_group'].replace('NoCoupon_Matched', 'NoCoupon')

def calculate_smd(df, group_col, treatment_name, control_name, features):
    smd_results = {}
    df_treat = df[df[group_col] == treatment_name]
    df_ctrl = df[df[group_col] == control_name]
    for col in features:
        x_t = df_treat[col]
        x_c = df_ctrl[col]
        pooled_std = np.sqrt((x_t.var() + x_c.var()) / 2)
        smd_results[col] = abs((x_t.mean() - x_c.mean()) / pooled_std) if pooled_std != 0 else 0
    return pd.Series(smd_results)


df_balanced_pool = pd.concat([matched_coupon, matched_control])

df_smd_after = pd.DataFrame({
    'SMD_Coupon_vs_NoCoupon_AFTER': calculate_smd(df_balanced_pool, 'coupon_group', 'Coupon', 'NoCoupon_Matched', top_10_features)
})

display(df_smd_after)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns


sns.set_theme(style="whitegrid")


fig, ax = plt.subplots(figsize=(10, 6))



sns.kdeplot(data=df_psm[df_psm['coupon_group']=='NoCoupon'], x='ps_score', label='NoCoupon (Control)', fill=True, alpha=0.1, color='b', ax=ax)


sns.kdeplot(data=df_psm[df_psm['coupon_group']=='Coupon'], x='ps_score', label='Coupon (Treatment)', fill=True, alpha=0.1, color='orange', ax=ax)


ax.set_title("Common Support Check: Binary Propensity Score for 'Coupon'", fontsize=14, fontweight='bold')
ax.set_xlabel("Propensity Score (Probability of getting Coupon)", fontsize=12, )
ax.set_ylabel("Density", fontsize=12)
ax.legend(fontsize=11)


ax.set_xlim(-0.02, 0.10)

plt.tight_layout()
plt.show()

## Stage 1 | T-Learner and ENB

Train separate LightGBM models and compute the difference in predicted churn. The original heuristic sets uplift to zero when the coupon propensity exceeds 0.65; this is not a general common-support test. Coupon ENB uses a value of TWD 149 and a cost of TWD 35.

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb


test_df = df_test.copy()

if 'coupon_group_temp' in test_df.columns:
    df_no_coupon_universe = test_df[test_df['coupon_group_temp'] == 'NoCoupon']
elif 'coupon_group' in test_df.columns:
    df_no_coupon_universe = test_df[test_df['coupon_group'] == 'NoCoupon']
else:

    test_df['coupon_temp'] = test_df['last_plan_list_price'] - test_df['last_actual_paid']
    df_no_coupon_universe = test_df[(test_df['last_actual_paid'] > 0) & (test_df['coupon_temp'] <= 0)]

sample_size = 10000

if len(df_no_coupon_universe) >= sample_size:
    df_decision = df_no_coupon_universe.sample(n=sample_size, random_state=42).copy()
else:
    pass
    df_decision = df_no_coupon_universe.copy()

features = json.loads((DATA / 'prototype_feature_columns.json').read_text())
target_col = 'is_churn'


lgb_params = {
    'objective': 'binary',
    'metric': 'binary_logloss',
    'learning_rate': 0.05,
    'verbose': -1,
    'random_state': 42
}


df_nc_train = df_train_balanced[df_train_balanced['coupon_group'] == 'NoCoupon']
df_cp_train = df_train_balanced[df_train_balanced['coupon_group'] == 'Coupon']

model_NC = lgb.train(lgb_params, lgb.Dataset(df_nc_train[features], label=df_nc_train[target_col]), num_boost_round=100)

model_Coupon = lgb.train(lgb_params, lgb.Dataset(df_cp_train[features], label=df_cp_train[target_col]), num_boost_round=100)




top_10_features_psm = [
    'last_is_auto_renew', 'auto_renew_streak', 'last_payment_plan_days',
    'last_is_cancel', 'days_since_last_log', 'auto_renew_rate',
    'last_days_since_trans',
    '90d_num_25_count', 'total_cancellations', 'registered_via'
]

X_decision_psm = df_decision[top_10_features_psm].apply(pd.to_numeric, errors='coerce').fillna(0)


X_decision_scaled = scaler.transform(X_decision_psm)
decision_ps_probs = ps_model.predict_proba(X_decision_scaled)


coupon_idx = list(ps_model.classes_).index('Coupon')
df_decision['ps_score'] = decision_ps_probs[:, coupon_idx]



safe_causal_mask = df_decision['ps_score'] <= 0.65
n_outliers = len(df_decision) - safe_causal_mask.sum()
X_decision_model = df_decision[features]

df_decision['p_churn_NoCoupon'] = model_NC.predict(X_decision_model)
df_decision['p_churn_Coupon']   = model_Coupon.predict(X_decision_model)


df_decision['uplift_Score'] = df_decision['p_churn_NoCoupon'] - df_decision['p_churn_Coupon']



df_decision.loc[~safe_causal_mask, 'uplift_Score'] = 0.0

pass

In [ ]:
base_value = 149
coupon_cost = 35


df_decision['profit_NoCoupon'] = 0.0
df_decision['profit_Coupon']   = df_decision['uplift_Score'] * base_value - coupon_cost



df_decision['recommended_treatment'] = np.where(df_decision['profit_Coupon'] > 0, 'Coupon', 'NoCoupon')
df_decision['expected_net_benefit']  = np.where(df_decision['profit_Coupon'] > 0, df_decision['profit_Coupon'], 0.0)




roi_mask = (df_decision['expected_net_benefit'] <= 0)
df_decision.loc[roi_mask, 'recommended_treatment'] = 'NoCoupon'
df_decision.loc[roi_mask, 'expected_net_benefit']  = 0.0


df_decision['assigned_cost'] = np.where(df_decision['recommended_treatment'] == 'Coupon', coupon_cost, 0)
df_decision['final_uplift']  = np.where(df_decision['recommended_treatment'] == 'Coupon', df_decision['uplift_Score'], 0.0)

display(df_decision.groupby('recommended_treatment').agg(users=('recommended_treatment','size'), model_estimated_enb=('expected_net_benefit','sum')))

## Stage 2 | Free-trial prototype

Among users not selected for coupons, fit a new pair of models with a separate matching step. This version defines the historical free group using zero payment, positive listed price and no cancellation. It uses listed plan price as a monthly-value proxy and assumes three additional months of retention. These assumptions may overstate ENB, especially for non-monthly plans. The original policy is sequential, not a global optimization across all arms.

In [ ]:
HIGH_COUPON_COST = 149.0
FREE_ASSUMED_RETAINED_MONTHS = 3.0
BASE_VALUE = 149.0


X_features_psm = [
    "last_is_auto_renew",
    "auto_renew_streak",
    'last_days_since_trans',
    "last_payment_plan_days",
    "last_is_cancel",
    "days_since_last_log",
    "auto_renew_rate",
    "90d_num_25_count",
    "total_cancellations",
    "registered_via",
]
target_col = "is_churn"

df_free_trial = df_decision[
    df_decision["recommended_treatment"] == "NoCoupon"
].copy()
df_train = pd.read_parquet(DATA / "master_train_v2.parquet")




df_train["is_free_trial_history"] = np.where(
    (df_train["last_actual_paid"] == 0)
    & (df_train["last_plan_list_price"] > 0)
    & (df_train["last_is_cancel"] == 0),
    1,
    0,
)



df_train["coupon_temp"] = (
    df_train["last_plan_list_price"] - df_train["last_actual_paid"]
)
df_train["coupon_group_temp"] = np.select(
    [
        (df_train["last_actual_paid"] == 0),
        (df_train["last_actual_paid"] > 0) & (df_train["coupon_temp"] <= 0),
    ],
    ["Excluded", "NoCoupon"],
    default="Coupon",
)

df_s2_raw = df_train[
    (df_train["coupon_group_temp"] == "NoCoupon")
    | (df_train["is_free_trial_history"] == 1)
].copy()
df_s2_raw["is_treatment_stage2"] = np.where(
    df_s2_raw["is_free_trial_history"] == 1, 1, 0
)


scaler_s2 = StandardScaler()
X_s2_scaled = scaler_s2.fit_transform(
    df_s2_raw[X_features_psm].apply(pd.to_numeric, errors="coerce").fillna(0)
)

ps_model_s2 = LogisticRegression(solver="lbfgs", max_iter=500, random_state=42)
ps_model_s2.fit(X_s2_scaled, df_s2_raw["is_treatment_stage2"])
df_s2_raw["ps_score"] = ps_model_s2.predict_proba(X_s2_scaled)[:, 1]


caliper_s2 = 0.3 * df_s2_raw["ps_score"].std()
df_fr_treatment = df_s2_raw[df_s2_raw["is_treatment_stage2"] == 1].reset_index(
    drop=True
)
df_fr_control = df_s2_raw[df_s2_raw["is_treatment_stage2"] == 0].reset_index(
    drop=True
)

if len(df_fr_treatment) > 0 and len(df_fr_control) > 0:
    nn_s2 = NearestNeighbors(n_neighbors=1, algorithm="ball_tree")
    nn_s2.fit(df_fr_control[["ps_score"]])
    distances, indices = nn_s2.kneighbors(df_fr_treatment[["ps_score"]])


    valid_mask_s2 = distances.flatten() <= caliper_s2
    matched_treatment = df_fr_treatment[valid_mask_s2].copy()
    matched_control_idx = indices.flatten()[valid_mask_s2]
    matched_control = df_fr_control.iloc[matched_control_idx].copy()

    df_s2_balanced = pd.concat([matched_treatment, matched_control]).reset_index(
        drop=True
    )
    pass
    pass
else:
    df_s2_balanced = df_s2_raw.copy()
    pass




df_nc_train_s2 = df_s2_balanced[df_s2_balanced["is_treatment_stage2"] == 0]
df_fr_train_s2 = df_s2_balanced[df_s2_balanced["is_treatment_stage2"] == 1]


lgb_params = {
    "objective": "binary",
    "metric": "binary_logloss",
    "learning_rate": 0.05,
    "verbose": -1,
    "random_state": 42,
}


full_features = features

model_S2_NC = lgb.train(
    lgb_params,
    lgb.Dataset(df_nc_train_s2[full_features], label=df_nc_train_s2[target_col]),
    num_boost_round=100,
)
model_S2_FR = lgb.train(
    lgb_params,
    lgb.Dataset(df_fr_train_s2[full_features], label=df_fr_train_s2[target_col]),
    num_boost_round=100,
)

p_churn_S2_NC = model_S2_NC.predict(df_free_trial[full_features])
p_churn_S2_FR = model_S2_FR.predict(df_free_trial[full_features])


df_free_trial["uplift_free"] = p_churn_S2_NC - p_churn_S2_FR




if "last_plan_list_price" in df_free_trial.columns:
    monthly_value_col = "last_plan_list_price"
else:
    monthly_value_col = (
        "customer_value" if "customer_value" in df_free_trial.columns else None
    )

if monthly_value_col:
    df_free_trial["monthly_value_for_free_trial"] = pd.to_numeric(
        df_free_trial[monthly_value_col], errors="coerce"
    ).fillna(BASE_VALUE)
else:
    df_free_trial["monthly_value_for_free_trial"] = BASE_VALUE

df_free_trial.loc[
    df_free_trial["monthly_value_for_free_trial"] <= 0,
    "monthly_value_for_free_trial",
] = BASE_VALUE





df_free_trial["ENB_free_strategy"] = (
    df_free_trial["uplift_free"]
    * df_free_trial["monthly_value_for_free_trial"]
    * FREE_ASSUMED_RETAINED_MONTHS
    - HIGH_COUPON_COST
)


df_free_trial["free_trial_candidate"] = df_free_trial["ENB_free_strategy"] > 0

df_free_trial["free_trial_recommendation"] = np.where(
    df_free_trial["free_trial_candidate"],
    "Prototype candidate (ENB > 0)",
    "Not selected",
)


df_free_trial["required_retained_months_audit"] = np.where(
    df_free_trial["uplift_free"] > 0,
    HIGH_COUPON_COST
    / (
        df_free_trial["uplift_free"]
        * df_free_trial["monthly_value_for_free_trial"]
    ),
    np.inf,
)

# Aggregate-only summary of the two sequential stages.
counts = pd.Series({"Coupon": int((df_decision.recommended_treatment == "Coupon").sum()),
                    "Free Trial": int(df_free_trial.free_trial_candidate.sum()),
                    "NoCoupon": int((~df_free_trial.free_trial_candidate).sum())})
display(counts.to_frame("users"))
print("Stage 2 model-estimated ENB (TWD):", df_free_trial.loc[df_free_trial.free_trial_candidate, "ENB_free_strategy"].sum())

## Stage 2 | Covariate balance audit

In [ ]:
if 'df_s2_raw' not in globals() or 'df_s2_balanced' not in globals():
    raise FileNotFoundError("Run stage 2 before this audit.")


features_to_smd = [
    'last_is_auto_renew', 'auto_renew_streak', 'last_payment_plan_days',
    'last_is_cancel', 'days_since_last_log', 'auto_renew_rate',
    'last_days_since_trans', '90d_num_25_count', 'total_cancellations', 'registered_via'
]

smd_comparison_records = []

for col in features_to_smd:
    if col in df_s2_raw.columns and col in df_s2_balanced.columns:


        raw_t = pd.to_numeric(df_s2_raw[df_s2_raw['is_treatment_stage2'] == 1][col], errors='coerce').fillna(0)
        raw_c = pd.to_numeric(df_s2_raw[df_s2_raw['is_treatment_stage2'] == 0][col], errors='coerce').fillna(0)

        mean_rt, mean_rc = raw_t.mean(), raw_c.mean()
        var_rt, var_rc = raw_t.var(ddof=1), raw_c.var(ddof=1)
        pooled_std_raw = np.sqrt((var_rt + var_rc) / 2.0)
        raw_smd = (mean_rt - mean_rc) / pooled_std_raw if pooled_std_raw != 0 else 0.0


        match_t = pd.to_numeric(df_s2_balanced[df_s2_balanced['is_treatment_stage2'] == 1][col], errors='coerce').fillna(0)
        match_c = pd.to_numeric(df_s2_balanced[df_s2_balanced['is_treatment_stage2'] == 0][col], errors='coerce').fillna(0)

        mean_mt, mean_mc = match_t.mean(), match_c.mean()

        matched_smd = (mean_mt - mean_mc) / pooled_std_raw if pooled_std_raw != 0 else 0.0

        smd_comparison_records.append({
            'Feature_Name': col,
            'Raw_SMD_Before': raw_smd,
            'Matched_SMD_After': matched_smd,
            'Absolute_Raw_SMD': abs(raw_smd),
            'Absolute_Matched_SMD': abs(matched_smd),
            'Bias_Reduction_Rate': f"{abs(raw_smd - matched_smd) / abs(raw_smd):.2%}" if raw_smd != 0 else "0%"
        })


df_smd_final_report = pd.DataFrame(smd_comparison_records).sort_values(by='Absolute_Raw_SMD', ascending=False).reset_index(drop=True)


display(df_smd_final_report[['Feature_Name', 'Raw_SMD_Before', 'Matched_SMD_After', 'Absolute_Matched_SMD', 'Bias_Reduction_Rate']])
pass

## Historical prototype output (not rerun)

| Proposed assignment | Count |
|---|---:|
| Coupon | 305 |
| Free trial | 470 |
| No subsidy | 9,225 |
| Total | 10,000 |

The assignment counts are retained only to document the prototype. They do not demonstrate treatment benefit. The report's inconsistent no-subsidy count is reconciled here using the saved notebook output. Model-estimated financial gains are not promoted as validated results. The later diagnostics in notebook 03 explain why this policy should not be deployed.